# Train models in Google Colab

**Before running:**
1. *Runtime → Change runtime type → T4 GPU*.
2. Upload the `.npz` files from `data/processed/` to Google Drive, in `MyDrive/tfg/processed/`
   (`waveform_v2.npz` for fine-tuning, `logmel_win1024_v2.npz` for the CNN, `panns_cnn14_v2.npz` for the linear model).

The runs, the MLflow database (`colab.db`) and the models are saved in `MyDrive/tfg/mlflow/`.

## 1. Drive, code and dependencies

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/tfg"
REPO_DIR = "/content/acoustic-event-detection"
RESULTS_DIR = f"{DRIVE_DIR}/mlflow"

!pip install -q mlflow==3.17.2 panns_inference==0.1.1

%cd /content
!rm -rf {REPO_DIR}
!git clone -q https://github.com/armofz/acoustic-event-detection.git {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --oneline
!ln -s {DRIVE_DIR}/processed data/processed
!ls -lh data/processed

In [ ]:
import torch

print("torch", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 2. Train

The parameters depend on the model. Each run is one call to `train.py`.

In [ ]:
MODEL = "cnn14"                  # cnn, linear, cnn14 or mobilenetv1
FEATURES = "waveform_v2.npz"     # logmel_win1024_v2.npz (cnn), panns_cnn14_v2.npz (linear), waveform_v2.npz (fine-tuning)
MANIFEST = "manifest_v2.csv"
SEED = 666
LEARNING_RATE = 1e-4             # 1e-3 for cnn and linear, 1e-4 for fine-tuning
BATCH_SIZE = 16

!mkdir -p {RESULTS_DIR}
%cd {RESULTS_DIR}

!python {REPO_DIR}/src/train.py --model {MODEL} --seed {SEED} --manifest {REPO_DIR}/data/splits/{MANIFEST} \
--features {REPO_DIR}/data/processed/{FEATURES} --learning-rate {LEARNING_RATE} --batch-size {BATCH_SIZE}

# Results
!ls -lh {RESULTS_DIR} {RESULTS_DIR}/models